# Sesión 1: Preparación de datos con Power Query## Códigos M - Ejemplos y EjerciciosEste notebook contiene el código M de la presentación **Preparación de datos con Power Query** para copiar y pegar directamente en el Editor de Power Query de Power BI.

---## SLIDE 19: Estructura de una consulta M### Concepto: Estructura básica let...inUna consulta en M siempre tiene esta estructura:- **let**: Define cada paso intermedio- **in**: Indica el resultado final a retornar

```mlet    paso1 = 1,    paso2 = 2,    paso3 = paso1 + paso2in    paso3```**Explicación:**- `paso1`: Define el valor 1- `paso2`: Define el valor 2- `paso3`: Suma los dos valores anteriores- `in paso3`: Retorna el resultado del paso3 (3)

## Ejemplo de carga de CSV con estructura completaEste es un ejemplo realista de cómo cargar y transformar un archivo CSV desde Power Query:

```mlet    origen = Csv.Document(        File.Contents("C:\\datos\\ventas.csv"),        [Delimiter=",", Columns=5, Encoding=65001]    ),    con_encabezados = Table.PromoteHeaders(        origen,        [PromoteAllScalars=true]    ),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {            {"ID", Int64.Type},            {"Fecha", type date},            {"Monto", type number},            {"Cliente", type text},            {"Estado", type text}        }    )in    cambiar_tipos```**Explicación:**1. `origen`: Carga el archivo CSV con delimitador "," y codificación UTF-82. `con_encabezados`: Promociona la primera fila como encabezados3. `cambiar_tipos`: Convierte cada columna al tipo de dato correcto4. `in cambiar_tipos`: Retorna la tabla con tipos ya convertidos

## SLIDE 20: Comandos y funciones comunes### 1. Filtrar filas (Table.SelectRows)

```mlet    origen = Excel.Workbook(File.Contents("ventas.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    filtrar_activos = Table.SelectRows(        con_encabezados,        each [Estado] = "Activo"    )in    filtrar_activos```**Explicación:**- `Table.SelectRows()`: Filtra filas según una condición- `each [Estado] = "Activo"`: Mantiene solo filas donde la columna "Estado" es "Activo"- `each`: Palabra clave que aplica la condición a cada fila

### 2. Eliminar columnas (Table.RemoveColumns)

```mlet    origen = Excel.Workbook(File.Contents("datos.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    eliminar_columnas = Table.RemoveColumns(        con_encabezados,        {"ColumnaAuxiliar", "TempoDatos", "NoUsada"}    )in    eliminar_columnas```**Explicación:**- `Table.RemoveColumns()`: Elimina las columnas especificadas- Las columnas se especifican entre llaves `{}`- Útil para limpiar datos intermedios antes de cargar al modelo

### 3. Transformar texto (Text.Upper, Text.Trim)

```mlet    origen = Excel.Workbook(File.Contains("clientes.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    limpiar_espacios = Table.TransformColumns(        con_encabezados,        {"Nombre", Text.Trim, type text}    ),    a_mayusculas = Table.TransformColumns(        limpiar_espacios,        {"Nombre", Text.Upper, type text}    )in    a_mayusculas```**Explicación:**- `Text.Trim`: Elimina espacios iniciales y finales- `Text.Upper`: Convierte a mayúsculas- `Text.Lower`: Convierte a minúsculas- `Table.TransformColumns()`: Aplica transformación a una columna

### 4. Convertir texto a fecha (Date.FromText)

```mlet    origen = Csv.Document(        File.Contents("transacciones.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    convertir_fecha = Table.TransformColumns(        con_encabezados,        {"FechaTransaccion", each Date.FromText(_, "yyyy-MM-dd"), type date}    )in    convertir_fecha```**Explicación:**- `Date.FromText()`: Convierte texto a tipo fecha- `"yyyy-MM-dd"`: Especifica el formato esperado del texto- Otros formatos: `"dd/MM/yyyy"`, `"MM-dd-yyyy"`, etc.

### 5. Valores únicos (List.Distinct)

```mlet    origen = Excel.Workbook(File.Contents("datos.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    sin_duplicados = Table.Distinct(        con_encabezados,        Comparer.OrdinalIgnoreCase    )in    sin_duplicados```**Explicación:**- `Table.Distinct()`: Elimina filas duplicadas- `Comparer.OrdinalIgnoreCase`: No distingue mayúsculas/minúsculas

### 6. Redondeo de números (Number.Round)

```mlet    origen = Excel.Workbook(File.Contents("precios.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    redondear_precios = Table.TransformColumns(        con_encabezados,        {"PrecioUnitario", each Number.Round(_, 2), type number}    )in    redondear_precios```**Explicación:**- `Number.Round(_, 2)`: Redondea a 2 decimales- El guion bajo `_` representa el valor actual de cada celda

### 7. División de texto (Text.Split)

```mlet    origen = Csv.Document(        File.Contents("empleados.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    dividir_nombre = Table.AddColumn(        con_encabezados,        "Nombre",        each Text.Split([NombreCompleto], " "){0},        type text    ),    dividir_apellido = Table.AddColumn(        dividir_nombre,        "Apellido",        each Text.Split([NombreCompleto], " "){1},        type text    )in    dividir_apellido```**Explicación:**- `Text.Split()`: Divide un texto por un delimitador- `{0}`: Selecciona el primer elemento (índice 0)- `{1}`: Selecciona el segundo elemento- Útil para separar nombres completos

## SLIDE 31-32: Detección y tratamiento de valores perdidos o nulos### 1. Filtrar valores nulos

```mlet    origen = Excel.Workbook(File.Contents("clientes.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    sin_nulos = Table.SelectRows(        con_encabezados,        each [Telefono] <> null and [Telefono] <> ""    )in    sin_nulos```**Explicación:**- `<> null`: Diferente de nulo- `<> ""`: Diferente de texto vacío- `and`: Ambas condiciones deben cumplirse

### 2. Reemplazar valores nulos por defecto

```mlet    origen = Excel.Workbook(File.Contents("ventas.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    reemplazar_nulos = Table.ReplaceValue(        con_encabezados,        null,        "Sin Datos",        Replacer.ReplaceValue,        {"Observaciones"}    )in    reemplazar_nulos```**Explicación:**- `Table.ReplaceValue()`: Reemplaza valores en columnas especificadas- `null`: Valor a buscar- `"Sin Datos"`: Valor con el que reemplazar- `{"Observaciones"}`: Aplica solo en esta columna

### 3. Usar columnas condicionales para imputación basada en reglas

```mlet    origen = Excel.Workbook(File.Contents("transacciones.xlsx"), null, true),    tabla = origen{0}[Data],    con_encabezados = Table.PromoteHeaders(tabla),    imputar_por_region = Table.AddColumn(        con_encabezados,        "MontoImputado",        each if [Monto] = null then                if [Region] = "Norte" then 1000                else if [Region] = "Sur" then 800                else 900            else                [Monto],        type number    )in    imputar_por_region```**Explicación:**- `if...then...else`: Estructura condicional anidada- Verifica primero si Monto es nulo- Si es nulo, imputa según la Región- Si no es nulo, mantiene el valor original

### 4. Agrupar por categoría y calcular promedio (para imputación robusta)

```mlet    origen = Csv.Document(        File.Contents("diagnósticos.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"Edad", Int64.Type}    ),    promedio_por_diagnostico = Table.Group(        cambiar_tipos,        {"Diagnostico"},        {{"EdadPromedio", each List.Average([Edad]), type number}}    )in    promedio_por_diagnostico```**Explicación:**- `Table.Group()`: Agrupa registros por columna- `{"Diagnostico"}`: Columna por la que agrupar- `List.Average()`: Calcula el promedio de valores en Edad

## SLIDE 33: Normalización y transformación de fechas### 1. Cambiar tipo de dato a fecha

```mlet    origen = Csv.Document(        File.Contents("eventos.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    convertir_a_fecha = Table.TransformColumnTypes(        con_encabezados,        {"FechaEvento", type date}    )in    convertir_a_fecha```

### 2. Detectar fechas fuera de rango

```mlet    origen = Csv.Document(        File.Contents("registros_medicos.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"FechaRegistro", type date}    ),    validar_rango = Table.AddColumn(        cambiar_tipos,        "FechaValida",        each if [FechaRegistro] < #date(1950, 1, 1) or [FechaRegistro] > Date.From(DateTime.LocalNow()) then                false            else                true,        type logical    ),    solo_fechas_validas = Table.SelectRows(        validar_rango,        each [FechaValida] = true    )in    solo_fechas_validas```**Explicación:**- `#date(año, mes, día)`: Formato de fecha literal en M- `DateTime.LocalNow()`: Obtiene la fecha/hora actual- Filtra registros con fechas anteriores a 1950 o posteriores a hoy

### 3. Extraer componentes de fecha

```mlet    origen = Csv.Document(        File.Contents("transacciones.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"Fecha", type date}    ),    extraer_año = Table.AddColumn(        cambiar_tipos,        "Año",        each Date.Year([Fecha]),        type number    ),    extraer_mes = Table.AddColumn(        extraer_año,        "Mes",        each Date.Month([Fecha]),        type number    ),    extraer_dia = Table.AddColumn(        extraer_mes,        "Dia",        each Date.Day([Fecha]),        type number    )in    extraer_dia```**Explicación:**- `Date.Year()`: Extrae el año- `Date.Month()`: Extrae el mes (1-12)- `Date.Day()`: Extrae el día del mes

### 4. Calcular diferencias entre fechas

```mlet    origen = Csv.Document(        File.Contents("pacientes.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"FechaNacimiento", type date, "FechaRegistro", type date}    ),    calcular_dias = Table.AddColumn(        cambiar_tipos,        "DiasDesdeRegistro",        each Duration.Days([FechaRegistro] - [FechaNacimiento]),        type number    )in    calcular_dias```**Explicación:**- `[FechaRegistro] - [FechaNacimiento]`: Resta fechas (resultado es duración)- `Duration.Days()`: Convierte la duración a número de días

## SLIDE 34: Transformación de texto y limpieza semántica### 1. Transformación a mayúsculas/minúsculas

```mlet    origen = Csv.Document(        File.Contents("ciudades.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    a_mayusculas = Table.TransformColumns(        con_encabezados,        {"NombreCiudad", Text.Upper, type text}    ),    a_minusculas = Table.TransformColumns(        a_mayusculas,        {"Pais", Text.Lower, type text}    )in    a_minusculas```**Funciones disponibles:**- `Text.Upper`: Mayúsculas- `Text.Lower`: Minúsculas- `Text.Proper`: Primera letra mayúscula

### 2. Eliminar espacios dobles o iniciales

```mlet    origen = Csv.Document(        File.Contents("clientes.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    limpiar_espacios = Table.TransformColumns(        con_encabezados,        {"Nombre", Text.Trim, type text}    )in    limpiar_espacios```**Funciones disponibles:**- `Text.Trim`: Elimina espacios al inicio y final- `Text.TrimStart`: Solo al inicio- `Text.TrimEnd`: Solo al final

### 3. Sustituir caracteres (guiones por espacios)

```mlet    origen = Csv.Document(        File.Contents("direcciones.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    reemplazar_caracteres = Table.ReplaceValue(        con_encabezados,        "-",        " ",        Replacer.ReplaceText,        {"Direccion"}    )in    reemplazar_caracteres```**Explicación:**- `Replacer.ReplaceText`: Busca todas las ocurrencias- `-`: Carácter a buscar- ` `: Carácter de reemplazo

## SLIDE 34: Agrupaciones y operaciones por categoría### 1. Contar registros por grupo

```mlet    origen = Csv.Document(        File.Contents("ventas.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"Fecha", type date, "Monto", type number}    ),    contar_por_sucursal = Table.Group(        cambiar_tipos,        {"Sucursal"},        {{"TotalVentas", Table.RowCount, Int64.Type}}    )in    contar_por_sucursal```

### 2. Sumar valores numéricos por grupo

```mlet    origen = Csv.Document(        File.Contents("ventas.csv"),        [Delimiter=","]    ),    con_encabezados = Table.PromoteHeaders(origen),    cambiar_tipos = Table.TransformColumnTypes(        con_encabezados,        {"Fecha", type date, "Monto", type number}    ),    suma_por_sucursal = Table.Group(        cambiar_tipos,        {"Sucursal"},        {{"MontoTotal", each List.Sum([Monto]), type number}}    )in    suma_por_sucursal```

## Referencias rápidas### Tipos de datos en M- `type text`: Texto- `type number`: Número (decimal)- `Int64.Type`: Número entero- `Decimal.Type`: Número con precisión decimal- `type date`: Fecha- `type time`: Hora- `type datetime`: Fecha y hora- `type logical`: Booleano (verdadero/falso)### Operadores de comparación- `=`: Igual- `<>`: Diferente- `<`: Menor que- `>`: Mayor que- `<=`: Menor o igual- `>=`: Mayor o igual- `and`: Y (ambas condiciones)- `or`: O (al menos una condición)- `not`: Negación### Notas finales- Todos los códigos anteriores son ejemplos que puedes **copiar y pegar** directamente en el Editor de Power Query- Reemplaza los nombres de archivos y columnas según tu caso específico- M es **case-sensitive**: `[Fecha]` ≠ `[fecha]`- El panel "Pasos Aplicados" muestra cada transformación. Puedes hacer clic en cualquiera para editar